<a href="https://colab.research.google.com/github/Rageel-28/244107020136-Machine-Learning/blob/main/JS04/JS04_TUGAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab - Regresi Biaya Medis (Medical Cost Personal Datasets)

**Mata Kuliah:** Pembelajaran Mesin (JS04 - Regresi)

**Instruksi Umum:**
1. Menggunakan dataset *Medical Cost Personal Datasets* (umum dikenal sebagai `insurance.csv`).
2. Menggunakan Python dengan NumPy, Pandas, Matplotlib, dan Scikit-Learn untuk analisis regresi dan SVR.

**Tugas:**
1. Identifikasi variabel bebas (fitur) dan variabel target (biaya medis personal / `charges`).
2. Bagi dataset menjadi data latih dan data uji dengan proporsi yang sesuai.
3. Lakukan feature scaling jika diperlukan.
4. Buat model *multiple linear regression* menggunakan Scikit-Learn.
5. Latih model pada data latih dan lakukan prediksi pada data uji.
6. Evaluasi model dengan R-squared, MSE, dan MAE.
7. Ulangi langkah 4 dengan model SVR, termasuk eksperimen *hyperparameter tuning*.

> **Catatan:** Download dataset dari halaman Tugas Lab pada modul GitBook, lalu simpan dengan nama `insurance.csv` pada folder yang sama dengan notebook ini. Dataset ini memiliki kolom: `age`, `sex`, `bmi`, `children`, `smoker`, `region`, `charges`.

## Langkah 1 - Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Langkah 2 - Load Dataset

In [ ]:
data = pd.read_csv('insurance.csv')
data.head()


In [ ]:
data.shape


In [ ]:
data.info()


In [ ]:
data.describe()


## Langkah 3 - Eksplorasi Data (Deskripsi Variabel)

**Deskripsi dataset:** dataset berisi data biaya medis personal dari perusahaan asuransi, dengan variabel:
- `age`: usia tertanggung
- `sex`: jenis kelamin (male/female)
- `bmi`: indeks massa tubuh
- `children`: jumlah anak/tanggungan
- `smoker`: status merokok (yes/no)
- `region`: wilayah tempat tinggal di AS
- `charges`: **variabel target** — biaya medis yang ditagihkan oleh asuransi

Variabel bebas (fitur): `age`, `sex`, `bmi`, `children`, `smoker`, `region`.
Variabel target: `charges`.

In [ ]:
# Cek missing value
data.isnull().sum()


In [ ]:
# Visualisasi distribusi charges
plt.figure(figsize=(8, 4))
sns.histplot(data['charges'], bins=30, kde=True)
plt.title('Distribusi Biaya Medis (charges)')
plt.show()


In [ ]:
# Visualisasi hubungan antar fitur numerik dengan charges
sns.pairplot(data, x_vars=['age', 'bmi', 'children'], y_vars='charges', height=4, kind='scatter')
plt.show()


In [ ]:
# Pengaruh status merokok terhadap biaya
plt.figure(figsize=(6, 4))
sns.boxplot(x='smoker', y='charges', data=data)
plt.title('Biaya Medis Berdasarkan Status Merokok')
plt.show()


## Langkah 4 - Preprocessing (Encoding Variabel Kategorik)

Variabel `sex`, `smoker`, dan `region` bersifat kategorik sehingga perlu diubah menjadi numerik (encoding) sebelum digunakan pada model regresi.

In [ ]:
data_encoded = pd.get_dummies(data, columns=['sex', 'smoker', 'region'], drop_first=True)
data_encoded.head()


In [ ]:
# Memisahkan variabel bebas (X) dan variabel target (y)
X = data_encoded.drop('charges', axis=1)
y = data_encoded['charges']


## Langkah 5 - Split Data Latih dan Data Uji

Membagi dataset dengan proporsi 80:20 untuk data latih dan data uji.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Jumlah data latih: {X_train.shape[0]}')
print(f'Jumlah data uji: {X_test.shape[0]}')


## Langkah 6 - Feature Scaling

Melakukan penskalaan pada fitur numerik (`age`, `bmi`, `children`) agar berada pada rentang yang sebanding. Fitting scaler hanya pada data latih, lalu diterapkan ke data uji untuk menghindari *data leakage*.

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'bmi', 'children']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train_scaled.head()


## Langkah 7 - Multiple Linear Regression

Membuat, melatih, dan memprediksi menggunakan model *multiple linear regression*.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)


In [ ]:
# Melihat koefisien tiap fitur
coef_df = pd.DataFrame({
    'Fitur': X_train_scaled.columns,
    'Koefisien': lr_model.coef_
}).sort_values('Koefisien', key=abs, ascending=False)
coef_df


## Langkah 8 - Evaluasi Model Multiple Linear Regression

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print('=== Multiple Linear Regression ===')
print('R-squared :', r2_lr)
print('MSE       :', mse_lr)
print('MAE       :', mae_lr)
print('RMSE      :', rmse_lr)


In [ ]:
# Visualisasi nilai aktual vs prediksi
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Charges Aktual')
plt.ylabel('Charges Prediksi')
plt.title('Aktual vs Prediksi - Multiple Linear Regression')
plt.show()


## Langkah 9 - Support Vector Regression (SVR)

Mengulangi pemodelan dengan SVR. Karena SVR sensitif terhadap skala, target `y` juga perlu discaling.

In [ ]:
from sklearn.svm import SVR

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

svr_model = SVR(kernel='rbf', C=100, epsilon=0.1)
svr_model.fit(X_train_scaled, y_train_scaled)

y_pred_svr_scaled = svr_model.predict(X_test_scaled)
y_pred_svr = y_scaler.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()


## Langkah 10 - Hyperparameter Tuning SVR (GridSearchCV)

Mencoba beberapa kombinasi hyperparameter (`C`, `epsilon`, `gamma`) untuk mendapatkan model SVR terbaik.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [1, 10, 100, 1000],
    'epsilon': [0.01, 0.1, 0.5],
    'gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(
    SVR(kernel='rbf'), param_grid, cv=5,
    scoring='r2', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train_scaled)

print('Parameter terbaik:', grid_search.best_params_)
print('Best CV R-squared :', grid_search.best_score_)


In [ ]:
best_svr = grid_search.best_estimator_

y_pred_best_svr_scaled = best_svr.predict(X_test_scaled)
y_pred_best_svr = y_scaler.inverse_transform(y_pred_best_svr_scaled.reshape(-1, 1)).ravel()


## Langkah 11 - Evaluasi Model SVR

In [ ]:
r2_svr = r2_score(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

r2_best_svr = r2_score(y_test, y_pred_best_svr)
mse_best_svr = mean_squared_error(y_test, y_pred_best_svr)
mae_best_svr = mean_absolute_error(y_test, y_pred_best_svr)

print('=== SVR (default: kernel=rbf, C=100, epsilon=0.1) ===')
print('R-squared :', r2_svr)
print('MSE       :', mse_svr)
print('MAE       :', mae_svr)
print()
print('=== SVR (setelah hyperparameter tuning) ===')
print('R-squared :', r2_best_svr)
print('MSE       :', mse_best_svr)
print('MAE       :', mae_best_svr)


## Langkah 12 - Perbandingan Model

In [ ]:
perbandingan = pd.DataFrame({
    'Model': ['Multiple Linear Regression', 'SVR (default)', 'SVR (tuned)'],
    'R-squared': [r2_lr, r2_svr, r2_best_svr],
    'MSE': [mse_lr, mse_svr, mse_best_svr],
    'MAE': [mae_lr, mae_svr, mae_best_svr]
})
perbandingan


In [ ]:
perbandingan.set_index('Model')[['R-squared']].plot(kind='bar', legend=False, figsize=(6,4))
plt.ylabel('R-squared')
plt.title('Perbandingan R-squared Antar Model')
plt.xticks(rotation=15)
plt.show()


## Analisis dan Kesimpulan

- Fitur `smoker` umumnya memberikan pengaruh paling besar terhadap `charges`, terlihat dari koefisien pada model *multiple linear regression* maupun boxplot pada tahap eksplorasi data.
- Bandingkan nilai R-squared, MSE, dan MAE pada tabel `perbandingan` di atas untuk menentukan model mana (*multiple linear regression*, SVR default, atau SVR hasil tuning) yang memberikan performa terbaik pada data uji.
- Jika SVR belum mengungguli regresi linier, pertimbangkan memperluas `param_grid` pada `GridSearchCV`, atau mencoba kernel lain seperti `linear` atau `poly`.

